<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/10_text_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Analysis

In [1]:
import nltk

In [2]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')
nltk.download('tagsets_json')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package tagsets_json to /root/nltk_data...
[nltk_data]   Package tagsets_json is already up-to-date!


True

In [3]:
import polars as pl
from nltk.corpus import stopwords, wordnet
from nltk.stem import (
    PorterStemmer,
    LancasterStemmer,
    RegexpStemmer,
    SnowballStemmer,
    WordNetLemmatizer as wnl,
)
from nltk.tag import pos_tag

In [4]:
stop_words = set(stopwords.words('english'))
porter = PorterStemmer()
lancaster = LancasterStemmer()
snowball = SnowballStemmer('english')
regexp = RegexpStemmer('ing$|s$|e$|able$', min=4)
lemmatizer = wnl()

### Stemming: Comparing NLTK Stemmers

In [5]:
tokens_to_process = ['package', 'comfort-fit', 'running', 'shoes', 'arrived', 'damaged']
filtered_tokens = pl.DataFrame(
    {
        "tokens": [
            word for word in tokens_to_process
            if word.lower() not in stop_words
            ]
    }
    )
filtered_tokens = filtered_tokens.with_columns(
    pl.col("tokens").map_elements(
      porter.stem, return_dtype=pl.String).alias("porter_stemmed"),
    pl.col("tokens").map_elements(
      lancaster.stem, return_dtype=pl.String).alias("lancaster_stemmed"),
    pl.col("tokens").map_elements(
      snowball.stem, return_dtype=pl.String).alias("snowball_stemmed"),
)

display(filtered_tokens)

tokens,porter_stemmed,lancaster_stemmed,snowball_stemmed
str,str,str,str
"""package""","""packag""","""pack""","""packag"""
"""comfort-fit""","""comfort-fit""","""comfort-fit""","""comfort-fit"""
"""running""","""run""","""run""","""run"""
"""shoes""","""shoe""","""sho""","""shoe"""
"""arrived""","""arriv""","""ar""","""arriv"""
"""damaged""","""damag""","""dam""","""damag"""


### POS Tagging

In [6]:
tokens_to_process = nltk.word_tokenize("They refuse to permit us to obtain the refuse permit")
tagged_tokens = pl.DataFrame(
    pos_tag(tokens_to_process, tagset='universal'),
    schema=["token", "universal_pos_tag"],
    orient="row").join(
    pl.DataFrame(
        pos_tag(tokens_to_process),
        schema=["token", "penn_treebank"],
        orient="row"), on="token")
tagged_tokens

token,universal_pos_tag,penn_treebank
str,str,str
"""They""","""PRON""","""PRP"""
"""refuse""","""VERB""","""VBP"""
"""refuse""","""NOUN""","""VBP"""
"""to""","""PRT""","""TO"""
"""to""","""PRT""","""TO"""
…,…,…
"""the""","""DET""","""DT"""
"""refuse""","""VERB""","""NN"""
"""refuse""","""NOUN""","""NN"""


### Mapping Tags

In [7]:
def get_wordnet_pos(treebank_tag):
   """
   Maps Treebank POS tags to WordNet POS tags.
   """
   if treebank_tag.startswith('J'):
       return wordnet.ADJ
   elif treebank_tag.startswith('V'):
       return wordnet.VERB
   elif treebank_tag.startswith('N'):
       return wordnet.NOUN
   elif treebank_tag.startswith('R'):
       return wordnet.ADV
   else:
       # As a default, assume noun if the tag is not recognized
       return wordnet.NOUN

### Lemmatization Example

In [8]:
sentence = "The children are running and playing in the beautiful gardens"
# 1. Tokenize sentence
tokens = nltk.word_tokenize(sentence)
# 2. Tag the tokens
tagged_tokens = pos_tag(nltk.word_tokenize(sentence))
# 3. Filter the stop words
filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
# 4. Lemmatize the tokens using POS tags
# Stop words are not skipped here just for demonstration
lemmas = [
    lemmatizer.lemmatize(token, get_wordnet_pos(tag))
    for token, tag in tagged_tokens
    # for token, tag in pos_tag(filtered_tokens)
]
print("Tokens: ")
print(tokens)
print("Filtered Tokens: ")
print(filtered_tokens)
print("Lemmas: ")
print(lemmas)
print("Filtered Lemmas: ")
print(list(lemma for lemma in lemmas if lemma.lower() not in stop_words))

pl.DataFrame({
    "token": tokens ,
    "lemma": lemmas,
    "is_stop_words": [word.lower() in stop_words for word in nltk.word_tokenize(sentence)],
    })

Tokens: 
['The', 'children', 'are', 'running', 'and', 'playing', 'in', 'the', 'beautiful', 'gardens']
Filtered Tokens: 
['children', 'running', 'playing', 'beautiful', 'gardens']
Lemmas: 
['The', 'child', 'be', 'run', 'and', 'play', 'in', 'the', 'beautiful', 'garden']
Filtered Lemmas: 
['child', 'run', 'play', 'beautiful', 'garden']


token,lemma,is_stop_words
str,str,bool
"""The""","""The""",true
"""children""","""child""",false
"""are""","""be""",true
"""running""","""run""",false
"""and""","""and""",true
"""playing""","""play""",false
"""in""","""in""",true
"""the""","""the""",true
"""beautiful""","""beautiful""",false


## Text Data Preparation Example

In [9]:
df = pl.read_csv("/content/global-cart.csv")
for idx, comment in enumerate(df['FeedbackText']):
  print(f"{idx+1}. {comment}\n")

1. The Pro-Grade Blender is a beast! It's powerful and quiet. Delivery was also incredibly fast, arrived in one day.

2. My package with the Comfort-Fit Running Shoes arrived damaged, and the box was completely crushed. Disappointed.

3. I had to return the blender. It was much larger than expected and didn't fit on my counter. The return was easy.

4. Are the Comfort-Fit shoes waterproof? Need to know before I buy. Your support chat is offline.

5. Fast shipping is great, but the Pro-Grade Blender's lid doesn't seem to seal properly. Seems like a defect.



In [10]:
def prep_doc(
    doc,
    *,
    lemmatizer=None,
    stemmer=None,
    exclude_stop_words=True,
    ):
  tokens = nltk.word_tokenize(doc)

  filtered_tokens = tokens
  lemmas = None
  stemmed_tokens = None

  if exclude_stop_words:
    filtered_tokens = [
        word for word in tokens if word.lower() not in stop_words
        ]

  if lemmatizer:
    lemmas = [
      lemmatizer.lemmatize(token, get_wordnet_pos(tag))
      for token, tag in pos_tag(filtered_tokens)
    ]

  if stemmer:
    stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]

  return {
    "tokens": tokens,
    "filtered_tokens": filtered_tokens,
    "lemmas": lemmas,
    "stemmed_tokens": stemmed_tokens,
  }

In [11]:
feedback = df.select(pl.col('FeedbackText')).to_series().to_list()

processed_docs = [prep_doc(doc, lemmatizer=lemmatizer, stemmer=snowball) for doc in feedback]

In [12]:
for k,v in processed_docs[0].items():
  print(f"{k}: {v}")
# print(processed_docs[-1])

tokens: ['The', 'Pro-Grade', 'Blender', 'is', 'a', 'beast', '!', 'It', "'s", 'powerful', 'and', 'quiet', '.', 'Delivery', 'was', 'also', 'incredibly', 'fast', ',', 'arrived', 'in', 'one', 'day', '.']
filtered_tokens: ['Pro-Grade', 'Blender', 'beast', '!', "'s", 'powerful', 'quiet', '.', 'Delivery', 'also', 'incredibly', 'fast', ',', 'arrived', 'one', 'day', '.']
lemmas: ['Pro-Grade', 'Blender', 'beast', '!', "'s", 'powerful', 'quiet', '.', 'Delivery', 'also', 'incredibly', 'fast', ',', 'arrive', 'one', 'day', '.']
stemmed_tokens: ['pro-grad', 'blender', 'beast', '!', "'s", 'power', 'quiet', '.', 'deliveri', 'also', 'incred', 'fast', ',', 'arriv', 'one', 'day', '.']
